# 02 — Logistic Regression baseline

A simple, interpretable benchmark. It uses a reduced feature set, median imputation, frequency encoding for categoricals, scaling, and `class_weight='balanced'`. Compare results with the later tree models; do not deploy this by default.

In [ ]:
!pip -q install pandas pyarrow scikit-learn
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

DATA = Path('/content/drive/MyDrive/ieee_fraud/processed')
ARTIFACTS = Path('/content/drive/MyDrive/ieee_fraud/artifacts'); ARTIFACTS.mkdir(exist_ok=True)
train, valid, test = [pd.read_parquet(DATA / f'{x}.parquet') for x in ['train', 'validation', 'test']]
metadata = json.loads((DATA / 'preparation_metadata.json').read_text())
features = metadata['feature_columns']
# Keep this baseline manageable and avoid near-unique IDs.
features = [c for c in features if train[c].nunique(dropna=False) < 1000]
X_train, y_train = train[features].copy(), train.isFraud
X_valid, y_valid = valid[features].copy(), valid.isFraud
numeric = X_train.select_dtypes(include=np.number).columns.tolist()
categorical = [c for c in features if c not in numeric]
print(len(features), len(numeric), len(categorical))

In [ ]:
# Training-only frequency encoding avoids huge sparse one-hot matrices.
def frequency_encode(train_x, other_x, columns):
    train_x, other_x = train_x.copy(), other_x.copy()
    for col in columns:
        mapping = train_x[col].fillna('MISSING').astype(str).value_counts(normalize=True)
        train_x[col] = train_x[col].fillna('MISSING').astype(str).map(mapping).fillna(0).astype('float32')
        other_x[col] = other_x[col].fillna('MISSING').astype(str).map(mapping).fillna(0).astype('float32')
    return train_x, other_x

X_train, X_valid = frequency_encode(X_train, X_valid, categorical)
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=500, class_weight='balanced', solver='saga', n_jobs=-1, random_state=42))
])
pipeline.fit(X_train, y_train)
probability = pipeline.predict_proba(X_valid)[:, 1]
metrics = {'model': 'logistic_regression', 'roc_auc': float(roc_auc_score(y_valid, probability)), 'pr_auc': float(average_precision_score(y_valid, probability))}
print(metrics)
(ARTIFACTS / 'logistic_regression_metrics.json').write_text(json.dumps(metrics, indent=2))